# **Cell 1: Install & Import**

In [27]:
!pip install pdfplumber

import pdfplumber
import pandas as pd
import numpy as np
import os
import json
import glob
import re

print("All libraries loaded ✅")

All libraries loaded ✅


# **Cell 2: Upload PDF**

In [28]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"Uploaded: {filename} ✅")

Saving budget speech 2083.pdf to budget speech 2083 (2).pdf
Uploaded: budget speech 2083 (2).pdf ✅


# **Cell 3 — PDF Diagnostic (read before parsing)**

In [29]:
filename = "budget speech 2083.pdf"

with pdfplumber.open(filename) as pdf:
    total_pages = len(pdf.pages)
    print(f"Total pages: {total_pages}")
    print("=" * 50)

    # Looking at first 5 pages — understand structure
    for i in range(min(5, total_pages)):
        page = pdf.pages[i]
        text = page.extract_text()
        print(f"\n--- PAGE {i+1} ---")
        if text:
            print(text[:500])  # First 500 chars only
        else:
            print("(no extractable text on this page)")
        print()

Total pages: 93

--- PAGE 1 ---
(no extractable text on this page)


--- PAGE 2 ---
सङ्घ‍ीय‍संसदको‍संयक्तु ‍बैठकमा‍
अर्मथ न्त्री‍डा.‍स्वर्णमथ ‍वाग्लेद्वारा‍प्रस्ततु
आर्र्कथ ‍वर्‍थ 208३/8४‍को‍बजेट‍वक्तव्य
नेपाल‍सरकार
अर्‍थ मन्त्रालय
208३‍जेठ‍15
www.mof.gov.np


--- PAGE 3 ---
प्रर्तर्नर्िसभाका‍सम्माननीय‍सभामखु ‍महोदय
राष्ट्रिय‍सभाका‍सम्माननीय‍अध्यक्ष‍महोदय,
१. आिर्ुनक‍नेपालका‍प्रर्म‍अर्मथ न्त्री‍श्री‍सवु ण‍थ शमशेरबाट‍कररब‍७५‍वर्अथ र्घ‍(२००८‍
साल‍माघ‍२१‍गते)‍प्रारम्भ‍भएको‍मलु कु को‍आय‍र‍व्यय‍ष्ट्रववरण‍पेश‍गने‍प्रणालीलाई‍
र्नरन्त्तरता‍ददंदै‍यो‍सम्मार्नत‍सदनमा‍उर्भिँदा‍गौरवार्न्त्वत‍छु।‍"जनताको‍जीवनस्तर‍बेहतर"‍
पाने‍तत्कालीन‍उद्देश्य‍आज‍पर्न‍उर्िकै ‍सान्त्दर्भकथ ‍छ।‍यस‍अवसरमा‍नागररक‍स्वतन्त्रता,
लोकतार्न्त्रक‍ अर्िकार, सामार्जक‍ न्त्याय‍ र‍ समन्नु त‍ नेपालका‍ लार्ग‍ भएक


--- PAGE 4 ---
छ।‍यस‍यारामा‍रचनात्मक‍सझु ाव‍र‍सहकायकथ ा‍लार्ग‍सङ्घ‍ीय‍संसद, राज्यका‍सबै‍
र्नकाय, नागररक‍समाज‍तर्ा‍सम्पूण‍थ नेपाली‍जनतासमक्ष‍सरकारका‍तफथ बाट‍हाददथक‍अनरु ोि‍
गदथछु।
५. मलु कु को‍ ष्ट्रवद्यमान‍ आर्र्कथ ‍ अवस्

# **Cell 4 — Checking for Tables**

In [30]:
with pdfplumber.open(filename) as pdf:
    print("Checking first 10 pages for tables...\n")

    for i in range(min(10, total_pages)):
        page = pdf.pages[i]
        tables = page.extract_tables()
        if tables:
            print(f"PAGE {i+1}: Found {len(tables)} table(s)")
            # Show first table structure
            print(f"  First table preview: {tables[0][:3]}")
        else:
            print(f"PAGE {i+1}: No tables")

Checking first 10 pages for tables...

PAGE 1: No tables
PAGE 2: No tables
PAGE 3: No tables
PAGE 4: No tables
PAGE 5: No tables
PAGE 6: No tables
PAGE 7: No tables
PAGE 8: No tables
PAGE 9: No tables
PAGE 10: No tables


# **Cell 5: Extract All Text First**
This is a "broken dataset" moment so the fastest and most reliable fix for today is: we build the dataset manually from the budget speech text using Claude itself to extract the numbers — then save as CSV

In [31]:
filename = "budget speech 2083.pdf"

all_text = []

with pdfplumber.open(filename) as pdf:
    total_pages = len(pdf.pages)
    print(f"Extracting text from {total_pages} pages...\n")

    for i in range(total_pages):
        page = pdf.pages[i]
        text = page.extract_text()
        if text and len(text.strip()) > 50:
            all_text.append({
                "page": i + 1,
                "text": text.strip()
            })
            print(f"Page {i+1}: {len(text)} chars ✅")
        else:
            print(f"Page {i+1}: skipped (image or empty)")

print(f"\nTotal pages with text: {len(all_text)}")

# Save all extracted text to a file
with open("budget_raw_text.txt", "w", encoding="utf-8") as f:
    for page in all_text:
        f.write(f"\n\n=== PAGE {page['page']} ===\n")
        f.write(page['text'])

print("Saved to budget_raw_text.txt ✅")

Extracting text from 93 pages...

Page 1: skipped (image or empty)
Page 2: 174 chars ✅
Page 3: 2311 chars ✅
Page 4: 1930 chars ✅
Page 5: 1829 chars ✅
Page 6: 2211 chars ✅
Page 7: 2145 chars ✅
Page 8: 2176 chars ✅
Page 9: 2525 chars ✅
Page 10: 2266 chars ✅
Page 11: 2345 chars ✅
Page 12: 1995 chars ✅
Page 13: 2415 chars ✅
Page 14: 2171 chars ✅
Page 15: 2336 chars ✅
Page 16: 2397 chars ✅
Page 17: 2087 chars ✅
Page 18: 1769 chars ✅
Page 19: 1913 chars ✅
Page 20: 1716 chars ✅
Page 21: 1610 chars ✅
Page 22: 1712 chars ✅
Page 23: 1590 chars ✅
Page 24: 1752 chars ✅
Page 25: 1747 chars ✅
Page 26: 1596 chars ✅
Page 27: 1779 chars ✅
Page 28: 1761 chars ✅
Page 29: 1818 chars ✅
Page 30: 1708 chars ✅
Page 31: 1665 chars ✅
Page 32: 1713 chars ✅
Page 33: 1649 chars ✅
Page 34: 1741 chars ✅
Page 35: 1700 chars ✅
Page 36: 1736 chars ✅
Page 37: 1667 chars ✅
Page 38: 1839 chars ✅
Page 39: 1818 chars ✅
Page 40: 1652 chars ✅
Page 41: 1904 chars ✅
Page 42: 1876 chars ✅
Page 43: 1882 chars ✅
Page 44: 1452 char

# **Cell 6 — Inspecting the Number-Heavy Pages**
Pages 52–74 have the most content (4000–5000 chars each) — those are almost certainly where the budget numbers live.

In [32]:
#  Look at pages 52-56 where numbers likely are
with pdfplumber.open(filename) as pdf:
    for i in [51, 52, 53, 54, 55]:  # 0-indexed, so page 52-56
        page = pdf.pages[i]
        text = page.extract_text()
        if text:
            print(f"\n{'='*60}")
            print(f"PAGE {i+1}")
            print('='*60)
            print(text[:1000])  # First 1000 chars


PAGE 52
आआयय (cid:67)(cid:67)ययययककोो (cid:1)(cid:1)ववववररणण
आआ(cid:21)(cid:21)थथकक(cid:16)(cid:16) ववषष (cid:16)(cid:16) 22008833//8844
अअननससुु ूचूचीी -- 11
(((cid:77)(cid:77)..ललााखखममाा))
22008833//8844 ककोो लल(cid:30)(cid:30)यय
22008811//8822 22008822//8833 (cid:31)(cid:31)ोोतत
(cid:1)(cid:1)ववववररणण ककोो ससंशंशोो(cid:21)(cid:21)धधतत
ककोो ययथथााथथ(cid:16)(cid:16) अअननममुु ाानन ररककमम ववैदैदेेििशशकक
ननेपेपाालल ससररककाारर
अअननददुु ाानन ऋऋणण
11 ररााजज(cid:39)(cid:39)वव ((22++33++44)) 1111,,9966,,1188,,9988 1133,,0000,,2222,,8800 1155,,8800,,3311,,8833 1155,,8800,,3311,,8833 00 00
2 कर 10,49,87,54 11,71,58,92 14,03,31,83 14,03,31,83 0 0
3 अ(cid:48)य राज(cid:39)व 1,28,94,48 1,22,49,45 1,77,00,00 1,77,00,00 0 0
4 (cid:1)व(cid:1)वध (cid:49)ा(cid:21)(cid:50) 17,36,96 6,14,43 0 0 0 0
55 ररााजज(cid:39)(cid:39)वव बबााँडँडफफााँटँटममााफफ(cid:16)(cid:16)तत 11,,4422,,8822,,7755 11,,6655,,0000,,0000 11,,7755,,0000,,0000 11,,7755,,0000,,0000 00 00
66 ससंघंघीीयय ससिि(cid:58)(cid:58)तत ककोोषषममाा द

# **Cell 7 — Searching Entire Document for Budget Numbers**

In [33]:
# Finding all lines containing crore/billion amounts
with open("budget_raw_text.txt", "r", encoding="utf-8") as f:
    content = f.read()

lines = content.split('\n')

# Keywords that appear near budget figures in Nepali
keywords = ['करोड', 'अर्ब', 'रुपैयाँ', '%', 'प्रतिशत', 'बजेट', 'विनियोजन']

print("Lines containing budget figures:\n")
count = 0
for line in lines:
    if any(kw in line for kw in keywords):
        print(line.strip())
        count += 1
        if count >= 50:  # Show first 50 matches
            break

print(f"\nTotal matching lines found: {count}")

Lines containing budget figures:

आर्र्कथ ‍वर्‍थ 208३/8४‍को‍बजेट‍वक्तव्य
फे ने‍दाष्ट्रयत्वका‍रूपमा‍ग्रहण‍गरेको‍छु।‍यो‍बजेट‍स्वदेश‍र‍ष्ट्रवदेशमा‍रहे बसेका‍सम्पूण‍थ
अष्ट्रवश्व‍ास‍चनु ौतीका‍रूपमा‍रहेका‍छन।् ‍यस‍बजेटको‍ध्येय‍यी‍चनु ौतीलाई‍अवसरमा‍
सरुु मा‍म‍आर्र्कथ ‍वर्‍थ २०८३/८४‍को‍बजेटमाफथ त‍बसाल्न‍खोर्जएको‍र्र्र्त‍र‍सवाङ्गथ ीण‍ष्ट्रवकासको‍
(क) न्त्यूनतम‍रु.‍२‍करोडसम्म‍कृ ष्ट्रर्‍तर्ा‍पशजु न्त्य‍उत्पादनका‍लार्ग‍प्रारर्म्भक‍पजिँु ी‍
आर्र्कथ ‍मूल्याङ्‍कन‍गरी‍बजेट‍तर्ा‍योजना‍प्रणालीमा‍समावेश‍गनेछौँ।‍
३१. यो‍बजेटले‍स्वास्र्थय‍सेवामा‍नीर्तगत‍फड्को‍माने‍लक्ष्य‍र्लएको‍छ।‍दक्ष‍स्वास्र्थय‍जनशर्क्त,‍
पर्श्च‍म‍राजमागकथ ो‍लार्ग‍रु.‍37‍अब‍थ 46‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍छु।
मागकथ ो‍लार्ग‍रु.‍17‍अब‍थ 64‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍छु।
सम्पन्न‍गनेछौँ।‍यस‍राजमागकथ ो‍लार्ग‍रु.‍4‍अब‍थ 65‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍
अब‍थ बजेट‍व्यवस्र्ा‍गरेको‍छु।
2‍अब‍थ 16‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍छु।
गन‍थ रु.‍1‍अब‍थ 46‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍छु।
रु.‍6‍अब‍थ 55‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍छु।
रु.‍6‍अब‍थ 25‍करोड‍ष्ट्रवर्नयोजन‍गरेको‍छु

# **What we Found till now:**


*   Page 53–55 has structured revenue tables with actual numbers in लाख (lakhs)
*   Cell 7 found real budget allocations like शिक्षा = रु. 218 अर्ब 30 करोड, स्वास्थ्य = रु. 101 अर्ब 95 करोड

This is exactly enough to build the dataset. Two sources:


*   This is exactly enough to build the dataset. Two sources:
*   Page 53–55 tables → revenue breakdown with numbers




# **Cell 8 — Extract Sector Allocations from Text**

In [34]:
# Final Fix — handle zero-width joiners + Nepali digits

with open("budget_raw_text.txt", "r", encoding="utf-8") as f:
    content = f.read()

# Step 1: Remove zero-width joiners and clean text
content_clean = content.replace('\u200d', '').replace('\u200c', '').replace('\u200b', '')

# Step 2: Convert Nepali/Devanagari digits to ASCII
nepali_digits = str.maketrans('०१२३४५६७८९', '0123456789')
content_clean = content_clean.translate(nepali_digits)

# Step 3: Now search for patterns
lines = content_clean.split('\n')
rows = []

for line in lines:
    line = line.strip()
    if not line:
        continue

    amount_crore = None

    # Pattern 1: X अर्ब Y करोड
    match1 = re.search(r'(\d+)\s*अर्?ब\s*(\d+)\s*करोड', line)
    # Pattern 2: X अर्ब only
    match2 = re.search(r'(\d+)\s*अर्?ब', line)
    # Pattern 3: X करोड only
    match3 = re.search(r'(\d+)\s*करोड', line)

    if match1:
        amount_crore = int(match1.group(1)) * 100 + int(match1.group(2))
    elif match2:
        amount_crore = int(match2.group(1)) * 100
    elif match3:
        amount_crore = int(match3.group(1))

    if amount_crore and amount_crore > 0:
        rows.append({
            "raw_line": line[:150],
            "amount_crore": amount_crore
        })

df_raw = pd.DataFrame(rows)
print(f"Total lines extracted: {len(df_raw)}")
print(df_raw.to_string())

Total lines extracted: 74
                                                                             raw_line  amount_crore
0   (क) न्त्यूनतमरु.2करोडसम्मकृ ष्ट्रर्तर्ापशजु न्त्यउत्पादनकालार्गप्रारर्म्भकपजिँु ी             2
1                          पर्श्चमराजमागकथ ोलार्गरु.37अबथ 46करोडष्ट्रवर्नयोजनगरेकोछु।            46
2                                    मागकथ ोलार्गरु.17अबथ 64करोडष्ट्रवर्नयोजनगरेकोछु।            64
3                     सम्पन्नगनेछौँ।यसराजमागकथ ोलार्गरु.4अबथ 65करोडष्ट्रवर्नयोजनगरेको            65
4                                                    2अबथ 16करोडष्ट्रवर्नयोजनगरेकोछु।            16
5                                             गनथ रु.1अबथ 46करोडष्ट्रवर्नयोजनगरेकोछु।            46
6                                                 रु.6अबथ 55करोडष्ट्रवर्नयोजनगरेकोछु।            55
7                                                 रु.6अबथ 25करोडष्ट्रवर्नयोजनगरेकोछु।            25
8             (ट) गल्छी-स्याफ्रुबेशी-रसवु ागिीसडकखण्डलाईस्तरोन्नर्तगनथ रु.

# **Cell 9 — Clean Budget Text**

In [38]:
#  Clean budget text and split into chunks

import requests
import json

# Clean the full text
with open("budget_raw_text.txt", "r", encoding="utf-8") as f:
    content = f.read()

content_clean = content.replace('\u200d', '').replace('\u200c', '').replace('\u200b', '')
nepali_digits = str.maketrans('०१२३४५६७८९', '0123456789')
content_clean = content_clean.translate(nepali_digits)

print(f"Total cleaned text length: {len(content_clean)} characters")

# Split into chunks
chunk_size = 3000
chunks = []
for i in range(0, len(content_clean), chunk_size):
    chunks.append(content_clean[i:i+chunk_size])

print(f"Total chunks to process: {len(chunks)}")

Total cleaned text length: 205957 characters
Total chunks to process: 69


In [41]:
# Delete old cache folder
import shutil
shutil.rmtree("budget_cache")
print("Deleted budget_cache ✅")

Deleted budget_cache ✅


# **Cell 10 — Extract Using Groq (Free)**

In [ ]:
# Cell 10 — Extract budget facts using Groq API with rate limit handling

import requests
import json
import time
import os
import glob

GROQ_API_KEY = "your-groq-key-here"

cache_dir = "budget_cache"
os.makedirs(cache_dir, exist_ok=True)

def extract_budget_facts(text_chunk, chunk_num):
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    prompt = f"""You are analyzing Nepal's government budget speech for fiscal year 2083/84 (2026/27).

Extract EVERY budget fact from this text chunk. Include:
- Sector allocations (education, health, roads, energy etc.)
- Salary or allowance changes (nurses, teachers, police etc.)
- Policy numbers (percentage changes, targets, counts)
- Infrastructure projects with amounts
- Social security amounts

Return ONLY a valid JSON array. No explanation, no markdown, no extra text.
Each item must have exactly these keys:
- "sector": one of: Education, Health, Roads, Energy, Agriculture, Water, ICT, Social Security, Tourism, Industry, Defense, Irrigation, Sports, Forest, Other
- "description": what the money is for in English, keep it short
- "amount_crore": number only, in crore NPR (1 अर्ब = 100 करोड, 1 लाख = 0.01 करोड), use 0 if no amount
- "amount_type": one of: allocation, revenue, salary_change, percentage, target, other
- "unit": one of: crore_npr, percent, count, other

If no budget facts exist in this chunk return an empty array: []

Text:
{text_chunk}

JSON array:"""

    body = {
        "model": "llama-3.3-70b-versatile",
        "max_tokens": 1500,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1
    }

    for attempt in range(3):  # retry up to 3 times
        try:
            response = requests.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers=headers,
                json=body,
                timeout=30
            )
            result = response.json()

            # Print actual error if choices missing
            if "choices" not in result:
                print(f"\n  API error: {result.get('error', {}).get('message', result)}")
                if "rate" in str(result).lower():
                    print(f"  Rate limited — waiting 60 seconds...")
                    time.sleep(60)
                    continue
                return []

            text = result["choices"][0]["message"]["content"].strip()
            text = text.replace("```json", "").replace("```", "").strip()

            # Fix common JSON issues
            if not text.startswith("["):
                text = "[]"

            facts = json.loads(text)
            return facts

        except json.JSONDecodeError as e:
            print(f"\n  JSON error chunk {chunk_num}: {e} — skipping")
            return []
        except Exception as e:
            print(f"\n  Error chunk {chunk_num}: {e}")
            time.sleep(5)

    return []


# Process all chunks — 3 second delay to stay within rate limits
print(f"Total chunks: {len(chunks)}")
print(f"Estimated time: ~{len(chunks) * 3 // 60} minutes\n")

for i, chunk in enumerate(chunks):
    cache_file = f"{cache_dir}/chunk_{i+1}.json"

    if os.path.exists(cache_file):
        print(f"Chunk {i+1}/{len(chunks)}: already cached ✅")
        continue

    print(f"Chunk {i+1}/{len(chunks)}: processing...", end=" ", flush=True)
    facts = extract_budget_facts(chunk, i+1)

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(facts, f, ensure_ascii=False)

    print(f"got {len(facts)} facts ✅")
    time.sleep(3)  # 3 second gap — stays within free tier limits

print("\nAll chunks processed!")

Total chunks: 69
Estimated time: ~3 minutes

Chunk 1/69: processing... got 0 facts ✅
Chunk 2/69: processing... got 10 facts ✅
Chunk 3/69: processing... got 0 facts ✅
Chunk 4/69: processing... got 16 facts ✅
Chunk 5/69: processing... 
  API error: Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kt9fc968e9cayd3jtwnd96hq` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 9927, Requested 2938. Please try again in 4.325s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing
  Rate limited — waiting 60 seconds...
got 10 facts ✅
Chunk 6/69: processing... 
  JSON error chunk 6: Expecting property name enclosed in double quotes: line 220 column 23 (char 5002) — skipping
got 0 facts ✅
Chunk 7/69: processing... got 14 facts ✅
Chunk 8/69: processing... got 10 facts ✅
Chunk 9/69: processing... 
  API error: Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kt9fc968e9cayd3jtwnd96hq` ser

KeyboardInterrupt: 

In [43]:
# Check what we have so far
import glob, json

cache_files = glob.glob("budget_cache/chunk_*.json")
cache_files.sort(key=lambda x: int(x.split("_")[-1].replace(".json", "")))

all_facts = []
for f in cache_files:
    with open(f, "r", encoding="utf-8") as fh:
        facts = json.load(fh)
        all_facts.extend(facts)

print(f"Cache files: {len(cache_files)}")
print(f"Total facts so far: {len(all_facts)}")

Cache files: 27
Total facts so far: 315


# **Cell 11 — Saving CSV and JSON Files**

In [45]:
# Cell 11 — Load all cached chunks into final CSV

cache_files = glob.glob("budget_cache/chunk_*.json")
cache_files.sort(key=lambda x: int(x.split("_")[-1].replace(".json", "")))

all_facts = []
for f in cache_files:
    with open(f, "r", encoding="utf-8") as fh:
        facts = json.load(fh)
        all_facts.extend(facts)

df_budget = pd.DataFrame(all_facts)
df_budget["fiscal_year_nepali"] = "2083/84"
df_budget["fiscal_year_english"] = "2026/27"
df_budget = df_budget.drop_duplicates(subset=["description"])
df_budget = df_budget[df_budget["description"].str.len() > 5]
df_budget.to_csv("nepal_budget_clean.csv", index=False)

sectors = sorted(df_budget["sector"].unique().tolist())
with open("sector_list.json", "w") as f:
    json.dump(sectors, f)

print(f"Saved nepal_budget_clean.csv — {len(df_budget)} rows ✅")
print(f"Saved sector_list.json — {len(sectors)} sectors ✅")

df_budget["amount_crore"] = pd.to_numeric(df_budget["amount_crore"], errors="coerce").fillna(0)

print("\nSector breakdown:")
print(df_budget.groupby("sector")["amount_crore"].sum().sort_values(ascending=False).to_string())
print(f"\nTotal budget captured: Rs. {df_budget['amount_crore'].sum():.0f} crore")

Saved nepal_budget_clean.csv — 297 rows ✅
Saved sector_list.json — 21 sectors ✅

Sector breakdown:
sector
Agriculture        3328.00
Roads               735.64
Irrigation          510.00
Health              396.10
Aviation            346.00
Social Security     227.00
Water               204.00
Industry            156.00
ICT                 126.00
Trade               118.00
Energy               85.54
Education            72.78
Forest               13.31
Sports                4.30
Tourism               4.00
Labor                 3.00
Culture               0.00
Defense               0.00
Finance               0.00
Environment           0.00
Other                 0.00

Total budget captured: Rs. 6330 crore
